# PD / HC inference on a new test set

Use this when new recordings arrive. Nothing needs retraining.

**Before running:** add the new `.txt` recordings as a Kaggle dataset and
attach it, then set `NEW_DATA` in cell 2 to its folder. The recordings must be
in the PhysioNet format the model was trained on: tab-separated, 19 columns
(time, L1-L8, R1-R8, TotalL, TotalR), 100 Hz.

Accelerator must be **GPU T4** — the weights use the `mamba_ssm` CUDA kernels.

In [ ]:
# 1) code + kernels
import os, subprocess, sys, importlib.util, torch
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
assert cap >= (7, 5), f'Need a T4 (sm_75+), got sm_{cap[0]}{cap[1]}. Settings -> Accelerator -> GPU T4'

if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Ahmadrezanourozii/Project-Time-Series.git',
                    '/kaggle/working/repo'], check=True)

if importlib.util.find_spec('mamba_ssm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation',
                    'causal-conv1d', 'mamba-ssm'], check=True)
import mamba_ssm; print('mamba_ssm', mamba_ssm.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
# 2) locate the model bundle and the recordings to score
import os

BUNDLE = None
for root, _, files in os.walk('/kaggle/input'):
    if 'manifest.json' in files and any(f.endswith('.pt') for f in files):
        BUNDLE = root; break
assert BUNDLE, 'Attach the dataset ah22reza/pd-mamba-bundle'

# <<< point this at the new recordings >>>
NEW_DATA = None
if NEW_DATA is None:                       # fall back to the 20 sample recordings
    for root, _, files in os.walk('/kaggle/input'):
        if any(f.endswith('.txt') for f in files) and 'sample_recordings' in root:
            NEW_DATA = root; break
assert NEW_DATA, 'Set NEW_DATA to the folder holding the new .txt recordings'

n = len([f for f in os.listdir(NEW_DATA) if f.endswith('.txt')])
print(f'bundle : {BUNDLE}\nnew data: {NEW_DATA}  ({n} recordings)')

In [ ]:
# 3) predict
!cd /kaggle/working/repo && python predict.py --bundle {BUNDLE} --input {NEW_DATA} \
    --foot both --out /kaggle/working/predictions.csv

import pandas as pd
df = pd.read_csv('/kaggle/working/predictions.csv')
print(df.to_string(index=False))
print(f"\n{(df.prediction == 'PD').sum()} PD / {(df.prediction == 'HC').sum()} HC")

**If the new recordings come with labels**, put a `labels.csv` with columns
`subject_id,label` (values `PD`/`HC`) next to them and add
`--labels <path>` to the command above; it will then print accuracy,
weighted precision/recall/F1 and AUC.

**Reading the scores.** `score_pd` is the mean probability over the
recording's windows, averaged across the five fold models. Expect roughly
75\% accuracy on genuinely unseen subjects — that is the figure the
subject-disjoint evaluation produced. Sensitivity runs higher than
specificity (86\% against 64\%), so false alarms are more likely than missed
patients.